# 🔬 G²SF — 로컬 JupyterHub 실행 노트북
> **대상 클래스**: `cable_gland` 단독 실행
> **환경**: conda `G2SF` (Python 3.9 / PyTorch 2.5.1 / CUDA 12.1 / RTX A6000)
> **최초 실행 순서**: Step 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8A → 9
> **재실행 순서**: Step 2 → 8B → 9
> ⚠️ Step 3~5는 최초 1회만 실행. 이미 완료된 경우 자동 스킵됨.

## Step 1 · 환경 점검

In [ ]:
import sys, subprocess
from pathlib import Path

print('=' * 55)
print(f'Python : {sys.version.split()[0]}')

r = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
     '--format=csv,noheader'],
    capture_output=True, text=True
)
if r.returncode == 0:
    for i, l in enumerate(r.stdout.strip().splitlines()):
        print(f'GPU {i} : {l}')
else:
    print('❌ GPU 없음')

r2 = subprocess.run(['conda', 'env', 'list'], capture_output=True, text=True)
print(f"G2SF 환경 : {'✅ 존재' if 'G2SF' in r2.stdout else '❌ Step 3에서 생성 필요'}")
print('=' * 55)

## Step 2 · 경로 설정
> ⚠️ 재실행 시에도 항상 이 셀을 먼저 실행하세요.

In [ ]:
from pathlib import Path
import subprocess, os

REPO_DIR    = Path('/home/jovyan/data/G2SF')
CODE_DIR    = REPO_DIR / 'G2SF_GITHUB'
CKPT_DIR    = CODE_DIR / 'Checkpoints'
RESULT_DIR  = CODE_DIR / 'Results'
DATASET_DIR = Path('/home/jovyan/data/datasets')
MVTEC_DIR   = Path('/home/jovyan/data/datasets/mvtec3d_preprocessed/dataset/MVTEC 3D')

CONDA_ENV   = 'G2SF'
G2SF_PIP    = '/opt/conda/envs/G2SF/bin/pip'
G2SF_PYTHON = '/opt/conda/envs/G2SF/bin/python'

for d in [CKPT_DIR, RESULT_DIR, DATASET_DIR, MVTEC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def conda_run(cmd, cwd=None, stream=False):
    full = ['conda', 'run', '--no-capture-output', '-n', CONDA_ENV] + cmd
    if stream:
        proc = subprocess.Popen(
            full, cwd=cwd,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1
        )
        for line in proc.stdout:
            print(line, end='')
        proc.wait()
        return proc
    return subprocess.run(full, cwd=cwd, capture_output=True, text=True)

print(f'REPO : {REPO_DIR}  {"✅" if REPO_DIR.exists() else "❌"}')
print(f'CODE : {CODE_DIR}  {"✅" if CODE_DIR.exists() else "❌"}')
print(f'DATA : {MVTEC_DIR}  {"✅" if MVTEC_DIR.exists() else "❌"}')
print('\n✅ 경로 설정 완료')

## Step 3 · conda G2SF 환경 생성
> 최초 1회만 실행. 이미 존재하면 자동 스킵됩니다.
> 소요시간: 약 5~10분

In [ ]:
r = subprocess.run(['conda', 'env', 'list'], capture_output=True, text=True)
if CONDA_ENV in r.stdout:
    print(f'✅ {CONDA_ENV} 환경 이미 존재 — Step 4로 이동')
else:
    print('🔨 환경 생성 중... (5~10분 소요)')
    result = subprocess.run([
        'conda', 'create', '-n', CONDA_ENV, '-y',
        'python=3.9',
        'pytorch==2.5.1', 'torchvision==0.20.1', 'pytorch-cuda=12.1',
        'cuda-nvcc=12.1.105', 'cuda-cudart-dev=12.1', 'cuda-libraries-dev=12.1',
        '-c', 'pytorch', '-c', 'nvidia/label/cuda-12.1.0', '-c', 'conda-forge',
        '--strict-channel-priority',
    ], capture_output=True, text=True)
    if result.returncode == 0:
        print('✅ 환경 생성 완료')
    else:
        print(f'❌ 실패\n{result.stderr[-500:]}')
        raise RuntimeError('conda 환경 생성 실패')

In [ ]:
# cuda-cccl 12.1 고정 (nv/target 헤더 충돌 방지)
# ⚠️ 미적용 시 pointnet2 빌드 시 fatal error: nv/target 발생
print('📦 cuda-cccl 12.1 고정 중...')
r = subprocess.run([
    'conda', 'install', '-n', CONDA_ENV, '-y',
    'cuda-cccl=12.1', 'cuda-cccl_linux-64=12.1',
    '-c', 'nvidia/label/cuda-12.1.0',
    '--no-update-deps'
], capture_output=True, text=True)
print('✅ 완료' if r.returncode == 0 else f'❌\n{r.stderr[-300:]}')

# 환경 검증
print('\n=== 환경 검증 ===')
for label, cmd in {
    'Python' : ['python', '--version'],
    'nvcc'   : ['nvcc', '--version'],
    'PyTorch': ['python', '-c',
                'import torch; print(torch.__version__, "| CUDA:", '
                'torch.version.cuda, "| GPU:", torch.cuda.is_available())'],
}.items():
    r = conda_run(cmd)
    out = (r.stdout + r.stderr).strip().splitlines()
    print(f'  {"✅" if r.returncode == 0 else "❌"} {label}: {out[-1][:80] if out else "(없음)"}')

## Step 4 · pointnet2 빌드 + 패키지 설치
> 소요시간: 빌드 약 5~8분, 패키지 설치 약 3~5분

In [ ]:
import shutil

r_check = conda_run(['python', '-c', 'import pointnet2_ops; print("ok")'])
if 'ok' in r_check.stdout:
    print('✅ pointnet2_ops 이미 설치됨')
else:
    # 이전 빌드 캐시 제거
    build_dir = REPO_DIR / 'pointnet2_ops_lib' / 'build'
    if build_dir.exists():
        shutil.rmtree(build_dir)

    CONDA_ENV_PATH = '/opt/conda/envs/G2SF'
    env = os.environ.copy()

    # CUDA 환경 변수 설정 (시스템 toolkit 없이 conda 환경만 사용)
    env['CUDA_HOME']          = CONDA_ENV_PATH
    env['CUDA_PATH']          = CONDA_ENV_PATH
    env['PATH']               = f"{CONDA_ENV_PATH}/bin:{env.get('PATH', '')}"
    env['LD_LIBRARY_PATH']    = f"{CONDA_ENV_PATH}/lib:{env.get('LD_LIBRARY_PATH', '')}"

    # nv/target 헤더 경로 명시적 지정
    INCLUDE = (f"{CONDA_ENV_PATH}/include:"
               f"{CONDA_ENV_PATH}/targets/x86_64-linux/include")
    env['CPATH']              = INCLUDE
    env['C_INCLUDE_PATH']     = INCLUDE
    env['CPLUS_INCLUDE_PATH'] = INCLUDE

    env['TORCH_CUDA_ARCH_LIST'] = '8.6'  # RTX A6000
    env['FORCE_CUDA']           = '1'
    env['MAX_JOBS']             = '4'

    for k in ['CONDA_PREFIX', 'CONDA_DEFAULT_ENV']:
        env.pop(k, None)

    print('🔨 pointnet2_ops 빌드 중... (5~8분)')
    proc = subprocess.Popen(
        [G2SF_PIP, 'install', str(REPO_DIR / 'pointnet2_ops_lib'),
         '--no-build-isolation', '--no-cache-dir'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=env
    )
    lines = []
    for line in proc.stdout:
        lines.append(line)
        if any(k in line.lower() for k in ['error', 'fatal', 'successfully']):
            print(line, end='')
    proc.wait()

    if proc.returncode == 0:
        print('✅ pointnet2_ops 빌드 성공!')
    else:
        print('❌ 빌드 실패 — 마지막 30줄:')
        print(''.join(lines[-30:]))
        raise RuntimeError('pointnet2 빌드 실패')

In [ ]:
# 필수 패키지 설치
pkgs = [
    'gdown', 'open3d', 'scikit-image', 'scikit-learn', 'tqdm', 'einops', 'timm',
    'opencv-python', 'matplotlib', 'omegaconf', 'termcolor', 'prettytable',
    'lightning', 'imgaug==0.4.0', 'faiss-gpu-cu12', 'rarfile', 'albumentations',
]
print('📦 패키지 설치 중...')
failed = []
for pkg in pkgs:
    r = subprocess.run([G2SF_PIP, 'install', pkg, '-q'],
                       capture_output=True, text=True)
    status = '✅' if r.returncode == 0 else '❌'
    print(f'  {status} {pkg}')
    if r.returncode != 0:
        failed.append(pkg)
print('\n✅ 완료' if not failed else f'\n❌ 실패한 패키지: {failed}')

## Step 5 · 소스코드 패치
> 5가지 버그를 일괄 패치합니다. 이미 적용된 경우 자동 스킵됩니다.

In [ ]:
import re

# 패치 1: imgaug numpy 2.0 호환
# numpy 2.0에서 np.sctypes 제거됨 → 명시적 타입 집합으로 교체
r = subprocess.run(
    [G2SF_PYTHON, '-c',
     'import importlib.util; s=importlib.util.find_spec("imgaug"); print(s.origin)'],
    capture_output=True, text=True
)
imgaug_main = Path(r.stdout.strip()).parent / 'imgaug.py'
if imgaug_main.exists():
    c = imgaug_main.read_text()
    if 'np.sctypes' in c:
        c = c.replace('NP_FLOAT_TYPES = set(np.sctypes["float"])',
                      'NP_FLOAT_TYPES = {np.float16, np.float32, np.float64}')
        c = c.replace('NP_INT_TYPES = set(np.sctypes["int"])',
                      'NP_INT_TYPES = {np.int8, np.int16, np.int32, np.int64}')
        c = c.replace('NP_UINT_TYPES = set(np.sctypes["uint"])',
                      'NP_UINT_TYPES = {np.uint8, np.uint16, np.uint32, np.uint64}')
        imgaug_main.write_text(c)
        print('✅ 패치 1: imgaug numpy 2.0 호환')
    else:
        print('✅ 패치 1: 이미 적용됨')

# 패치 2: train.py — UNSFusionModel import 오류 제거
train_py = CODE_DIR / 'Engine' / 'train.py'
c = train_py.read_text()
if 'UNSFusionModel' in c:
    train_py.write_text(c.replace(
        'from Fusion import FusionDataCollector, UNSFusionModel, FusionModelV2',
        'from Fusion import FusionDataCollector, FusionModelV2'
    ))
    print('✅ 패치 2: train.py UNSFusionModel 제거')
else:
    print('✅ 패치 2: 이미 적용됨')

# 패치 3: models.py — Checkpoint → Checkpoints 경로 수정
models_py = CODE_DIR / 'Model' / 'models.py'
c = models_py.read_text()
if './Checkpoint/' in c:
    models_py.write_text(c.replace('./Checkpoint/', './Checkpoints/'))
    print('✅ 패치 3: models.py 경로 수정')
else:
    print('✅ 패치 3: 이미 적용됨')

# 패치 4: Dataset/__init__.py — cable_gland 단독 실행 설정
dataset_init = CODE_DIR / 'Dataset' / '__init__.py'
c = dataset_init.read_text()
p = re.sub(
    r'def mvtec3d_classes\(\):\s+return \[.*?\]',
    'def mvtec3d_classes():
    return ["cable_gland"]',
    c, flags=re.DOTALL
)
if p != c:
    dataset_init.write_text(p)
    print('✅ 패치 4: cable_gland 단독 실행 설정')
else:
    print('✅ 패치 4: 이미 적용됨')

# 패치 5: ./Result → ./Results 전체 소스 일괄 수정
fixed = []
for f in CODE_DIR.rglob('*.py'):
    c = f.read_text()
    if './Result/' in c:
        f.write_text(c.replace('./Result/', './Results/'))
        fixed.append(f.name)
if fixed:
    print(f'✅ 패치 5: Result → Results 경로 수정 ({", ".join(fixed)})')
else:
    print('✅ 패치 5: 이미 적용됨')

print('\n✅ 모든 패치 완료')

## Step 6 · 가중치 및 데이터셋 준비
> **필요한 파일 목록**
> - `Checkpoints/dino_vitbase8_pretrain.pth` — DINOv2 (gdown 자동 다운)
> - `Checkpoints/pointmae_pretrain.pth` — PointMAE (gdown 자동 다운)
> - `Results/mvtec/` — G²SF 사전학습 가중치 (브라우저 직접 다운 후 Results/ 에 넣고 실행)
> - `datasets/.../cable_gland/` — MVTec3D (tar.xz 파일을 datasets/ 에 넣고 실행)

In [ ]:
import gdown

def download(fid, out, label):
    p = Path(out)
    if p.exists() and p.stat().st_size > 1e5:
        print(f'✅ {label}: 이미 존재 ({p.stat().st_size / 1e6:.0f} MB)')
        return True
    print(f'📥 {label} 다운로드 중...')
    gdown.download(id=fid, output=str(p), quiet=False)
    ok = p.exists() and p.stat().st_size > 1e5
    print(f'{"✅" if ok else "❌"} {label}')
    return ok

download('14vQqN4Do1Vnx2TZVJ16LAW81FRq3o-UQ',
         CKPT_DIR / 'dino_vitbase8_pretrain.pth', 'DINOv2')
download('14d04kH3bX2BbDEIJPI4MkRtolfQw_fds',
         CKPT_DIR / 'pointmae_pretrain.pth', 'PointMAE')

In [ ]:
# G²SF 사전학습 가중치 압축 해제
# ⚠️ 브라우저에서 직접 다운로드 후 Results/ 폴더에 넣어주세요:
#    https://drive.google.com/file/d/1rfnq2sncNONWsvohxjWcFKBi_BSDAieg/view
# ⚠️ 경고창에서 "그래도 다운로드" 반드시 클릭 (미클릭 시 31MB HTML 파일만 받아짐)
import zipfile, rarfile

g2sf_file = RESULT_DIR / 'g2sf_weights.zip'

if (RESULT_DIR / 'mvtec').exists():
    print('✅ G²SF 가중치 이미 압축 해제됨')
elif g2sf_file.exists():
    with open(g2sf_file, 'rb') as f:
        header = f.read(8)
    is_rar = header[:4] == b'Rar!'
    size_mb = g2sf_file.stat().st_size / 1e6
    print(f'파일 크기: {size_mb:.0f} MB | 형식: {"RAR" if is_rar else "ZIP"}')
    if size_mb < 50:
        raise ValueError('❌ 파일 불완전 — 브라우저에서 재다운로드 후 "그래도 다운로드" 클릭 필수')
    print('📂 압축 해제 중...')
    if is_rar:
        subprocess.run(['apt-get', 'install', '-y', 'unrar'], capture_output=True)
        with rarfile.RarFile(str(g2sf_file)) as rf:
            rf.extractall(str(RESULT_DIR))
    else:
        with zipfile.ZipFile(g2sf_file, 'r') as zf:
            zf.extractall(str(RESULT_DIR))
    print('✅ 완료')
else:
    print(f'❌ 파일 없음: {g2sf_file}')
    print('   https://drive.google.com/file/d/1rfnq2sncNONWsvohxjWcFKBi_BSDAieg/view')

In [ ]:
# cable_gland 데이터셋 tar 압축 해제
# tar.xz 파일을 /home/jovyan/data/datasets/ 폴더에 넣고 실행하세요
import tarfile

cable_dir = MVTEC_DIR / 'cable_gland'
if cable_dir.exists() and (cable_dir / 'test').exists():
    print(f'✅ cable_gland 데이터셋 이미 존재')
    print(f'   파일 수: {sum(1 for _ in cable_dir.rglob("*"))}개')
else:
    # tar 파일 자동 탐색 (하위 폴더 포함, .tar / .tar.gz / .tar.xz 지원)
    tar_candidates = (list(DATASET_DIR.rglob('*.tar')) +
                      list(DATASET_DIR.rglob('*.tar.gz')) +
                      list(DATASET_DIR.rglob('*.tar.xz')))
    cable_tars = [t for t in tar_candidates if 'cable' in t.name.lower()]

    if cable_tars:
        tar_path = cable_tars[0]
        print(f'📦 tar 파일 발견: {tar_path.name} ({tar_path.stat().st_size / 1e6:.0f} MB)')
        print('📂 압축 해제 중...')
        with tarfile.open(tar_path) as tf:
            tf.extractall(str(MVTEC_DIR), filter='data')
        print(f'✅ 완료 → {cable_dir}')
    else:
        print(f'❌ tar 파일을 찾을 수 없음')
        print(f'   {DATASET_DIR} 에 cable_gland tar.xz 파일을 넣어주세요')

# 최종 준비 상태 확인
print('\n=== 준비 상태 ===')
for name, path in {
    'DINOv2'          : CKPT_DIR / 'dino_vitbase8_pretrain.pth',
    'PointMAE'        : CKPT_DIR / 'pointmae_pretrain.pth',
    'G²SF 가중치'     : RESULT_DIR / 'mvtec',
    'cable_gland'     : MVTEC_DIR / 'cable_gland',
    'cable_gland/test': MVTEC_DIR / 'cable_gland' / 'test',
}.items():
    ok = Path(path).exists()
    print(f'  {"✅" if ok else "❌"} {name}')

## Step 7 · config 경로 패치

In [ ]:
import shutil as _sh

config_path = CODE_DIR / 'config_parse.py'
bak = config_path.with_suffix('.py.bak')
if not bak.exists():
    _sh.copy(config_path, bak)
    print(f'📋 백업 생성: {bak.name}')

# 논문 원본 경로 → 실제 데이터셋 경로로 교체
PATCHES = {
    '/home/xxx/MVTEC3D-AD_preprocessed': str(MVTEC_DIR),
    '/home/xxx/MVTEC3D-AD_original'    : str(MVTEC_DIR),
}
c = config_path.read_text()
changed = False
for old, new in PATCHES.items():
    if old in c:
        c = c.replace(old, new)
        print(f'✅ 경로 교체: {old}')
        changed = True
    else:
        print(f'✅ 이미 패치됨: {old}')

if changed:
    config_path.write_text(c)
print('\n✅ config 패치 완료')

## Step 8 · 추론 실행
> **8-A**: 최초 실행 — 피처 추출 + LSPN Fusion 모델 학습 + 추론
> **8-B**: 이후 실행 — 저장된 피처 캐시 재사용 (훨씬 빠름)
> 둘 중 하나만 실행하세요.

In [ ]:
# 실행 전 체크리스트
print('=' * 50)
ready = True
for name, path in {
    'main.py'          : CODE_DIR / 'main.py',
    'config'           : CODE_DIR / 'config_parse.py',
    'DINOv2'           : CKPT_DIR / 'dino_vitbase8_pretrain.pth',
    'PointMAE'         : CKPT_DIR / 'pointmae_pretrain.pth',
    'G²SF 가중치'      : RESULT_DIR / 'mvtec',
    'cable_gland/test' : MVTEC_DIR / 'cable_gland' / 'test',
}.items():
    ok = Path(path).exists()
    print(f'  {"✅" if ok else "❌"} {name}')
    if not ok:
        ready = False
print()
print('✅ 모든 준비 완료 — 8-A 또는 8-B를 실행하세요' if ready
      else '❌ 누락 항목 있음 — 위 Step을 먼저 완료하세요')

In [ ]:
# Step 8-A: 최초 실행 (피처 추출부터)
import re
import pandas as pd

print('🚀 G²SF 추론 시작 (최초 실행)...')
print('-' * 50)
proc = subprocess.Popen(
    ['conda', 'run', '--no-capture-output', '-n', CONDA_ENV,
     'python', 'main.py',
     '--load_feature',        'False',  # 피처 새로 추출
     '--load_fusion_dataset', 'True',
     '--load_fuser',          'True',
     '--dataset',             'mvtec',
     '--f_coreset',           '0.01',
     '--fusion_batch_size',   '1024',
     '--fusion_test_batch_size', '1024',
     '--num_workers',         '2'],
    cwd=str(CODE_DIR),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
modal_lines = []
for line in proc.stdout:
    print(line, end='')
    if 'sample-level auc' in line:
        modal_lines.append(line)
proc.wait()
print('-' * 50)
print('✅ 완료' if proc.returncode == 0 else f'❌ (exit {proc.returncode})')

In [ ]:
# Step 8-B: 이후 실행 (캐시 재사용)
import re
import pandas as pd

print('🚀 G²SF 추론 시작 (캐시 재사용)...')
print('-' * 50)
proc = subprocess.Popen(
    ['conda', 'run', '--no-capture-output', '-n', CONDA_ENV,
     'python', 'main.py',
     '--load_feature',        'True',   # 저장된 피처 재사용
     '--load_fusion_dataset', 'True',
     '--load_fuser',          'True',
     '--dataset',             'mvtec',
     '--f_coreset',           '0.01',
     '--fusion_batch_size',   '1024',
     '--fusion_test_batch_size', '1024',
     '--num_workers',         '2'],
    cwd=str(CODE_DIR),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
modal_lines = []
for line in proc.stdout:
    print(line, end='')
    if 'sample-level auc' in line:
        modal_lines.append(line)
proc.wait()
print('-' * 50)
print('✅ 완료' if proc.returncode == 0 else f'❌ (exit {proc.returncode})')

## Step 9 · 결과 확인
> Step 8 실행 후 바로 이어서 실행하세요. `modal_lines` 변수를 재사용합니다.

In [ ]:
import re
import pandas as pd

if not modal_lines:
    print('⚠️ modal_lines 없음 — Step 8을 먼저 실행하세요')
else:
    rows = []
    for line in modal_lines:
        m = re.match(
            r'Modal (\S+):\s+sample-level auc ([\d.]+), pixel-level auc ([\d.]+), '
            r'aupro ([\d.]+), aupro_0p01 ([\d.]+), aupro_0p1 ([\d.]+), aupro_0p05 ([\d.]+)',
            line
        )
        if m:
            rows.append({
                'Modal'    : m.group(1),
                'I-AUROC'  : float(m.group(2)),
                'P-AUROC'  : float(m.group(3)),
                'AUPRO@30%': float(m.group(4)),
                'AUPRO@1%' : float(m.group(5)),
                'AUPRO@10%': float(m.group(6)),
                'AUPRO@5%' : float(m.group(7)),
            })

    df = pd.DataFrame(rows).set_index('Modal')
    df = (df * 100).round(1)
    print('=== cable_gland 결과 ===')
    print(df.to_string())
    print()
    print('=== 논문 기준 (FUS) ===')
    paper = pd.DataFrame([{
        'Modal': '논문', 'I-AUROC': 97.1, 'P-AUROC': 99.7,
        'AUPRO@30%': '-', 'AUPRO@1%': 47.1, 'AUPRO@10%': '-', 'AUPRO@5%': '-'
    }]).set_index('Modal')
    print(paper.to_string())

In [ ]:
# 결함 타입별 Anomaly Map 시각화
import matplotlib.pyplot as plt

result_base = RESULT_DIR / 'mvtec' / 'Complete' / 'cable_gland'
if not result_base.exists():
    print('❌ 결과 없음 — Step 8을 먼저 실행하세요')
else:
    defect_types = [d.name for d in sorted(result_base.iterdir())
                    if d.is_dir() and d.name != 'fusion']

    n_cols = 3
    fig, axes = plt.subplots(len(defect_types), n_cols,
                             figsize=(18, 5 * len(defect_types)))
    if len(defect_types) == 1:
        axes = [axes]

    for row, dtype in enumerate(defect_types):
        imgs = sorted((result_base / dtype).glob('*.jpg'))[:n_cols]
        for col in range(n_cols):
            axes[row][col].axis('off')
            if col < len(imgs):
                axes[row][col].imshow(plt.imread(str(imgs[col])))
                axes[row][col].set_title(f'{dtype}/{imgs[col].name}', fontsize=9)

    plt.suptitle('G²SF — cable_gland 결함 시각화', fontsize=14, fontweight='bold')
    plt.tight_layout()
    out_path = REPO_DIR / 'result.png'
    plt.savefig(str(out_path), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'💾 저장됨: {out_path}')

## 🛠️ 트러블슈팅

| 증상 | 해결 |
|------|------|
| OOM (메모리 부족) | `--f_coreset 0.005 --fusion_batch_size 512` |
| pointnet2 빌드 실패 | `conda env remove -n G2SF -y` 후 Step 3 재실행 |
| `nv/target` 헤더 없음 | Step 3의 cuda-cccl 고정 셀 재실행 |
| `ModuleNotFoundError` | Step 4 패키지 설치 셀 재실행 |
| `./Result/` 경로 오류 | Step 5 패치 5 재실행 |
| 데이터셋 test 폴더 없음 | tar 파일 위치 확인 후 Step 6 재실행 |
| G²SF 가중치 31MB | 브라우저 재다운로드 후 "그래도 다운로드" 클릭 |
| 다른 클래스 추가 | Step 5 패치 4에서 `["cable_gland"]` 수정 |